# Semiconductor Device Simulation — Project Walkthrough

This notebook is a read-through of the project's results, in the order the
five sprints were built. It does not run any new simulation — it loads the
already-generated data in `results/processed/` and the figures in
`results/figures/`, and states what each one shows. For method and scope
detail behind any number here, see `docs/` — this notebook links to the
relevant file at each step rather than repeating it.

Research question (`docs/project_manual.md` §1): how do MOSFET channel
length, gate oxide thickness, and channel doping trade off drive current,
leakage current, and switching behavior in a low-power context?

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "results" / "processed"
FIGURES = ROOT / "results" / "figures"

## Sprint 1 — Physics validation (PN junction)

A 1D PN junction, solved numerically in DEVSIM, checked against closed-form
analytical formulas (`src/physics/analytical_pnjunction.py`). This is the
validation step that earns trust in the same DEVSIM physics stack before it
is used for the MOSFET. Full numbers: `docs/validation.md` §Sprint 1.

In [ ]:
pd.read_csv(PROCESSED / "pnjunction_validation_summary.csv")

In [ ]:
display(Image(filename=FIGURES / "pnjunction_potential_profile.png"))
display(Image(filename=FIGURES / "pnjunction_carrier_profiles.png"))
display(Image(filename=FIGURES / "pnjunction_forward_iv.png"))

**Reading these plots:** the potential profile shows a step across the
junction matching the analytical built-in potential to a relative error of
7.2e-14. The carrier-profile plot shows electron and hole concentration
crossing at `n_i = 1e10 cm\u207b\u00b3` exactly at the junction, as theory predicts.
The forward I-V curve gives an ideality factor ≈ 1.01 across the mid-bias
range — textbook diffusion-dominated diode behavior.

## Sprint 2 — Baseline 2D MOSFET

A planar NMOS (L = 1 µm, t_ox = 10 nm, N_A = 1e17 cm⁻³, N_D = 1e20 cm⁻³),
built directly in DEVSIM with a structured box mesh
(`src/device/mosfet_geometry.py`). Five sanity gates were checked —
monotonic I-V curves, no divergence, on-current above off-current, and an
actual inversion channel forming at the surface. Full numbers:
`docs/validation.md` §Sprint 2.

In [ ]:
pd.read_csv(PROCESSED / "baseline_mosfet_gate_summary.csv")

In [ ]:
display(Image(filename=FIGURES / "baseline_mosfet_id_vd.png"))
display(Image(filename=FIGURES / "baseline_mosfet_id_vg.png"))
display(Image(filename=FIGURES / "baseline_mosfet_electron_density.png"))

**Reading these plots:** the I_D-V_D curve shows a clean triode-to-
saturation transition. The I_D-V_G curve (log scale) shows a subthreshold
exponential rising into strong inversion across roughly 11 decades. The
electron-density plot is the important physical confirmation: a bright
n-type strip forms at the Si/oxide surface under the gate at V_G = 1.0 V,
connecting the two n+ source/drain regions — a real inversion channel, not
just a monotonic terminal current.

## Sprint 3 — Parameter sweeps

Three single-variable sweeps on top of the Sprint 2 baseline: channel
length, oxide thickness, channel doping. Each device is fully
characterized (`src/simulation/characterization.py`) to extract V_TH, g_m,
SS, I_ON, I_OFF (definitions: `docs/physics.md` §7). Full numbers and PASS
criteria: `docs/validation.md` §Sprint 3.

In [ ]:
print("Channel length sweep")
display(pd.read_csv(PROCESSED / "channel_length_sweep_metrics.csv"))
print("Oxide thickness sweep")
display(pd.read_csv(PROCESSED / "oxide_thickness_sweep_metrics.csv"))
print("Channel doping sweep")
display(pd.read_csv(PROCESSED / "doping_sweep_metrics.csv"))

In [ ]:
display(Image(filename=FIGURES / "channel_length_sweep_metrics.png"))
display(Image(filename=FIGURES / "oxide_thickness_sweep_metrics.png"))
display(Image(filename=FIGURES / "doping_sweep_metrics.png"))

**Reading these tables:** I_ON strictly decreases as channel length
increases (longer channel, more resistance). V_TH strictly increases and
I_ON strictly decreases as oxide thickness increases (thinner oxide gives
stronger gate coupling). V_TH strictly increases with channel doping. I_OFF
does **not** move meaningfully with any of the three parameters — this
model has no DIBL/GIDL/punch-through mechanism, so that null result is
expected, not a defect (`docs/limitations.md`). The channel-doping sweep
tops out at 1.5e17 cm⁻³, not the originally planned 1e18: higher doping
sends the solver into non-convergent oscillation, a genuine numerical
limit of this device model, documented in `docs/limitations.md`.

## Sprint 4 — Low-power trade-off scoring

Combines I_ON, I_OFF, and g_m from the 7 unique Sprint 3 configurations
into one weighted, normalized score
(`tradeoff_score = 0.3*ion_norm + 0.5*(1 - ioff_norm) + 0.2*gm_norm`).
Full method and interpretation: `docs/optimization.md`.

In [ ]:
scores = pd.read_csv(PROCESSED / "low_power_tradeoff_scores.csv")
scores.sort_values("tradeoff_score", ascending=False)

**Reading this table:** the ranking is driven almost entirely by I_ON and
g_m (which span ~6 and several decades respectively across the 7
configurations), not by I_OFF (which spans under 1 decade — inside this
solver's numerical noise floor, and even changes sign between rows). So
despite I_OFF carrying the largest weight (0.5) in the score definition,
it is not resolving a real leakage difference. `docs/optimization.md`
gives the full reasoning and names the better-supported answer to "which
configuration favors low power" using only the metrics this dataset
actually resolves (I_ON, g_m).

## Sprint 5 — Mesh sensitivity and reproducibility

The baseline device was re-characterized with every mesh spacing halved,
to check how much of each reported metric is a mesh artifact rather than a
converged physical result. Full numbers and interpretation:
`docs/validation.md` §Sprint 5, `docs/limitations.md`.

In [ ]:
pd.read_csv(PROCESSED / "mesh_sensitivity.csv")

**Reading this table:** V_TH, g_m, and subthreshold swing are
mesh-converged to well under 1.1% at this refinement level. I_ON is not
fully converged at the baseline mesh density — doubling resolution moves
it 5.7%, just over the 5% tolerance set for this check — so every reported
I_ON figure in this project is accurate to roughly one part in twenty, not
tighter. I_OFF's large relative difference between mesh densities is not
meaningful: both values sit inside the near-zero solver noise floor
already flagged in Sprint 3/4.

## Where to go next

- `docs/HANDOFF.md` — current project status in one page
- `docs/project_manual.md` — scope, physics model, repository layout
- `docs/physics.md` — every equation and constant used, with labeling
  convention (simulation input / assumed / analytical / calibrated)
- `docs/validation.md` — every numerical-vs-analytical and gate check, by
  sprint
- `docs/optimization.md` — the Sprint 4 trade-off method and written
  interpretation
- `docs/limitations.md` — every scope, modeling, and numerical limitation
  found along the way